In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import PSD_utils
from scipy.io import loadmat

In [ ]:
log2_xticks = np.array([4, 8, 16, 32, 64], dtype=float)    # x ticks for log2 scale

# Global plotting defaults
plt.rcParams.update({
    "figure.figsize": (7, 5),
    "axes.labelsize": 15,
    "axes.titlesize": 15,
    "xtick.labelsize": 15,
    "ytick.labelsize": 15,
    "legend.fontsize": 10,
    "lines.linewidth": 2,
    "grid.alpha": 0.4,
    "grid.linestyle": "--",
    "axes.grid": True,
})

FS_DHM = 115_200

In [ ]:
power = '0p35'
ROM_form = 'ABN'

r = 10

file_path = f'/disk/hyk049/WT_RomFit/{power}/'
filename = f"{file_path}{power}vpp_r={r}_{ROM_form}.mat"

print(filename)

data = loadmat(filename)
EFOM = data['EFOM']
EROM_1 = data['EROM_opt']

In [ ]:
# Compute PSD at the center point
i = 100     # centerpoint

k_FOM, psd_FOM, f_FOM, slope_FOM, *_ = PSD_utils.compute_PSD(EFOM[i,:], FS_DHM, 5, 30)
k_ROM, psd_ROM, f_ROM, slope_ROM, *_ = PSD_utils.compute_PSD(EROM_1[i,:], FS_DHM, 5, 30)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 5))

ax.plot(f_FOM, psd_FOM, color='C0', lw = 1, ls="-", label='experiment')
ax.plot(f_ROM, psd_ROM, color='C6', lw = 1, ls="--", label='rom')

ax.set_ylabel(r"$\log_{10}(\mathrm{PSD})$ [$\mu$m$^2$/(Hz)]", fontsize=18)

ax.set_xscale('log')
ax.set_xlabel("Frequency [Hz]", fontsize=18)

if ROM_form == 'ABN':
    plt.suptitle("$\mathrm{d}\mathbf{X}_r = [\mathbf{A}_r\mathbf{X}_r + \mathbf{B}_r\mathbf{u} + \mathbf{N}_r\mathbf{X}_r u]\mathrm{d}t + \mathbf{M}_r \mathrm{d}\mathbf{W}_t$")
elif ROM_form == 'AB':
    plt.suptitle("$\mathrm{d}\mathbf{X}_r = [\mathbf{A}_r\mathbf{X}_r + \mathbf{B}_r u]\mathrm{d}t + \mathbf{M}_r \mathrm{d}\mathbf{W}_t$")
elif ROM_form == 'AN':
    plt.suptitle("$\mathrm{d}\mathbf{X}_r = [\mathbf{A}_r\mathbf{X}_r + \mathbf{N}_r\mathbf{X}_r u]\mathrm{d}t + \mathbf{M}_r \mathrm{d}\mathbf{W}_t$")

plt.tight_layout()
plt.legend()
plt.show()

## Video of a 2-D data segment

The HDF5 file stores each time step as a separate dataset (`/main/00000`, `/main/00001`, ...), so the exporter streams frames instead of loading the whole 5760-sample block. The default selects 1800 frames across that block: at 30 fps, the result is exactly 60 seconds long. Only change `START_INDEX`, run the configuration cell, and then run the export cell.

In [ ]:
import importlib
from pathlib import Path
import capillary_video

# Reload edits made to capillary_video.py in this running notebook kernel.
capillary_video = importlib.reload(capillary_video)

DATA_PATH = Path(
    "/home/jonas/ucsd_thesis/11222025_j_0.04vpp_data_roi-none_cal-true.hdf5"
)
START_INDEX = 40000  # choose any value from 0 through 94_241
REMOVE_SPATIAL_MEAN = False
SCALE_MODE = "demeaned" if REMOVE_SPATIAL_MEAN else "absolute"
OUTPUT_PATH = Path(
    f"capillary_wave_start_{START_INDEX:06d}_{SCALE_MODE}.mp4"
)

In [ ]:
video_path = capillary_video.make_capillary_video(
    START_INDEX,
    data_path=DATA_PATH,
    output_path=OUTPUT_PATH,
    source_samples=5_760,
    video_seconds=60,
    fps=30,
    remove_spatial_mean=REMOVE_SPATIAL_MEAN,
)
video_path

Source frames [50,000, 55,760) span 49.991 ms.
Exporting 1,800 frames at 30 fps (60 s), using source steps of 3-4.
Playback is approximately 1,200x slower than the experiment.
Fixed color range: -4.820 to 4.820 microns
Rendered 1/1,800 frames
Rendered 180/1,800 frames
Rendered 360/1,800 frames
Rendered 540/1,800 frames
Rendered 720/1,800 frames
Rendered 900/1,800 frames
Rendered 1,080/1,800 frames
Rendered 1,260/1,800 frames
Rendered 1,440/1,800 frames
Rendered 1,620/1,800 frames
Rendered 1,800/1,800 frames
Saved /home/jonas/ucsd_thesis/CapillaryWaveTurbulence/capillary_wave_start_050000.mp4


PosixPath('/home/jonas/ucsd_thesis/CapillaryWaveTurbulence/capillary_wave_start_050000.mp4')